# **Gold layer Aggregated Tables**

### **Loading of Silver layer tables**

In [0]:
train = spark.table("grocery_catalog.silver.train_clean")
stores = spark.table("grocery_catalog.silver.stores_clean")
transactions = spark.table("grocery_catalog.silver.transactions_clean")
holidays = spark.table("grocery_catalog.silver.holidays_clean")
oil = spark.table("grocery_catalog.silver.oil_clean")

### **Creating Stores dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_store
USING DELTA AS
SELECT DISTINCT
store_nbr AS store_id,
city,
state,
type,
cluster
FROM `grocery_catalog`.silver.stores_clean;
select * from `grocery_catalog`.gold.dim_store;


store_id,city,state,type,cluster
11,Cayambe,Pichincha,B,6
45,Quito,Pichincha,A,11
31,Babahoyo,Los Rios,B,10
9,Quito,Pichincha,B,6
51,Guayaquil,Guayas,A,17
43,Esmeraldas,Esmeraldas,E,10
7,Quito,Pichincha,D,8
24,Guayaquil,Guayas,D,1
5,Santo Domingo,Santo Domingo De Los Tsachilas,D,4
14,Riobamba,Chimborazo,C,7


### **Creating Product dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dim_product
USING DELTA
AS
SELECT DISTINCT
    id AS product_id,
    family AS product_family
    
FROM `grocery_catalog`.silver.train_clean;
SELECT * FROM `grocery_catalog`.gold.dim_product

product_id,product_family
9,DELI
40,CLEANING
41,DAIRY
100,BABY CARE
109,EGGS
118,LADIESWEAR
124,PERSONAL CARE
131,SEAFOOD
135,BEVERAGES
156,MEATS


### **Creating date dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_date
USING DELTA AS
SELECT DISTINCT
date,
YEAR(date) AS year,
quarter(date) AS quarter,
MONTH(date) AS month,
WEEKOFYEAR(date) AS week,
DAYOFMONTH(date) AS day
FROM `grocery_catalog`.silver.train_clean;
SELECT * FROM `grocery_catalog`.gold.dim_date

date,year,quarter,month,week,day
1943-01-24,1943,1,1,3,24
1943-03-18,1943,1,3,11,18
1943-03-23,1943,1,3,12,23
1943-04-02,1943,2,4,13,2
1943-04-10,1943,2,4,14,10
1943-06-11,1943,2,6,23,11
1943-06-16,1943,2,6,24,16
1943-07-09,1943,3,7,27,9
1943-08-01,1943,3,8,30,1
1943-08-14,1943,3,8,32,14


### **Creating Promotion dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_promotion
USING DELTA AS
SELECT DISTINCT
onpromotion
FROM `grocery_catalog`.silver.train_clean;
SELECT * FROM `grocery_catalog`.gold.dim_promotion

onpromotion
32
57
172
157
108
195
183
245
259
672


### **Creating Holiday dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_holiday
USING DELTA AS
SELECT DISTINCT
date,
type,
locale,
locale_name,
description,
transferred
FROM `grocery_catalog`.silver.holidays_clean;
SELECT * FROM `grocery_catalog`.gold.dim_holiday

date,type,locale,locale_name,description,transferred
1947-10-08,HOLIDAY,NATIONAL,Ecuador,Independencia de Guayaquil,false
1946-04-16,EVENT,NATIONAL,Ecuador,Terremoto Manabi+1,false
1946-08-11,TRANSFER,NATIONAL,Ecuador,Traslado Primer Grito de Independencia,false
1944-06-23,HOLIDAY,LOCAL,Latacunga,Cantonizacion de Latacunga,false
1945-05-08,EVENT,NATIONAL,Ecuador,Dia de la Madre,false
1947-01-01,TRANSFER,NATIONAL,Ecuador,Traslado Primer dia del ano,false
1947-11-06,HOLIDAY,REGIONAL,Santa Elena,Provincializacion Santa Elena,false
1943-12-30,ADDITIONAL,NATIONAL,Ecuador,Primer dia del ano-1,false
1945-12-24,ADDITIONAL,NATIONAL,Ecuador,Navidad+1,false
1944-07-03,EVENT,NATIONAL,Ecuador,Mundial de futbol Brasil: Cuartos de Final,false


### **Creating Oil Price Dimension Table**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dim_oil
USING DELTA
AS
SELECT
    date,
    dcoilwtico AS oil_price
FROM grocery_catalog.silver.oil_clean;
SELECT * FROM `grocery_catalog`.gold.dim_oil

date,oil_price
1942-12-31,null
1943-01-01,93.14
1943-01-02,92.97
1943-01-03,93.12
1943-01-06,93.2
1943-01-07,93.21
1943-01-08,93.08
1943-01-09,93.81
1943-01-10,93.6
1943-01-13,94.27


### **Creating Sales Fact table**

In [0]:
%sql

CREATE OR REPLACE TABLE grocery_catalog.gold.fact_sales
USING DELTA
AS
SELECT
    t.date,
    t.store_nbr,
    t.family,
    t.sales AS sales,
    t.onpromotion,
    COALESCE(tr.transactions,0) AS transactions

FROM grocery_catalog.silver.train_clean t

LEFT JOIN grocery_catalog.silver.transactions_clean tr
ON t.store_nbr = tr.store_nbr
AND t.date = tr.date


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM `grocery_catalog`.gold.fact_sales
WHERE sales > 0
LIMIT 20



date,store_nbr,family,sales,onpromotion,transactions
1942-12-31,25,BEAUTY,2.0,0,770
1942-12-31,25,BEVERAGES,810.0,0,770
1942-12-31,25,BREAD/BAKERY,180.589,0,770
1942-12-31,25,CLEANING,186.0,0,770
1942-12-31,25,DAIRY,143.0,0,770
1942-12-31,25,DELI,71.09,0,770
1942-12-31,25,EGGS,46.0,0,770
1942-12-31,25,FROZEN FOODS,29.654999,0,770
1942-12-31,25,GROCERY I,700.0,0,770
1942-12-31,25,GROCERY II,15.0,0,770


### **Creating Aggregated tables for Business Insights**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dashboard_base
USING DELTA AS
SELECT
    d.date,
    d.year,
    d.quarter,
    d.month,
    d.week,
    d.day,

    s.store_id,
    s.city,
    s.state,
    s.type AS store_type,
    s.cluster,

    f.family AS product_family,

    f.onpromotion,
    h.holiday_type,
    h.locale,
    h.locale_name,
    h.holiday_description,
    h.transferred,

    o.oil_price,
    f.sales,
    f.transactions
FROM grocery_catalog.gold.fact_sales f
LEFT JOIN grocery_catalog.gold.dim_date d
    ON f.date = d.date
LEFT JOIN grocery_catalog.gold.dim_store s
    ON f.store_nbr = s.store_id
LEFT JOIN (
    SELECT
        date,
        FIRST(type) AS holiday_type,
        FIRST(locale) AS locale,
        FIRST(locale_name) AS locale_name,
        FIRST(description) AS holiday_description,
        FIRST(transferred) AS transferred
    FROM grocery_catalog.gold.dim_holiday
    GROUP BY date
) h
    ON f.date = h.date
LEFT JOIN grocery_catalog.gold.dim_oil o
    ON f.date = o.date;

num_affected_rows,num_inserted_rows


**dashboard_sales_trend**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dashboard_sales_trend
USING DELTA AS
SELECT
    year,
    quarter,
    month,
    week,
    day,
    date,
    SUM(sales) AS total_sales
FROM grocery_catalog.gold.dashboard_base
GROUP BY year, quarter, month, week, day, date;

num_affected_rows,num_inserted_rows


dashboard_store_performance

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dashboard_store_performance
USING DELTA AS
SELECT
    year,
    quarter,
    month,
    week,
    day,
    store_id,
    city,
    state,
    cluster,
    SUM(sales) AS total_sales
FROM grocery_catalog.gold.dashboard_base
GROUP BY year, quarter, month, week, day, store_id, city, state, cluster;

num_affected_rows,num_inserted_rows


**dashboard_product_insights**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dashboard_product_insights
USING DELTA AS
SELECT
    year,
    quarter,
    month,
    week,
    day,
    product_family,
    SUM(sales) AS total_sales
FROM grocery_catalog.gold.dashboard_base
GROUP BY year, quarter, month, week, day, product_family;

num_affected_rows,num_inserted_rows


**dashboard_promotion_impact**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dashboard_promotion_impact
USING DELTA AS
SELECT
    year,
    quarter,
    month,
    week,
    day,
    onpromotion,
    SUM(sales) AS total_sales
FROM grocery_catalog.gold.dashboard_base
GROUP BY year, quarter, month, week, day, onpromotion;

num_affected_rows,num_inserted_rows


**dashboard_holiday_sales**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dashboard_holiday_sales
USING DELTA AS
SELECT
    year,
    quarter,
    month,
    week,
    day,
    holiday_type,
    SUM(sales) AS total_sales
FROM grocery_catalog.gold.dashboard_base
GROUP BY year, quarter, month, week, day, holiday_type;

num_affected_rows,num_inserted_rows


**dashboard_customer_activity**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dashboard_customer_activity
USING DELTA AS
SELECT
    year,
    quarter,
    month,
    week,
    day,
    store_id,
    SUM(transactions) AS total_transactions,
    SUM(sales) AS total_sales
FROM grocery_catalog.gold.dashboard_base
GROUP BY year, quarter, month, week, day, store_id;

num_affected_rows,num_inserted_rows
